In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
!pip install torch-geometric -q
import os
os.environ['TORCH_V'] = torch.__version__.split('+')[0]
os.environ['CUDA_V'] = 'cu' + torch.version.cuda.replace('.', '')
!pip install pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH_V}+${CUDA_V}.html -q


⚠️ RESTART SESSION NOW (Run -> Restart Session), then continue from Cell 2.

In [ ]:
import os
REPO_ROOT = '/kaggle/working/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')

RAW_CACHE_DIR = '/kaggle/input/antenna-raw-cache-all-grids'
SEED_MASK_DIR = '/kaggle/input/antenna-seed-masks'
WORKING_DIR = '/kaggle/working'
os.makedirs(f'{WORKING_DIR}/data/processed/25x25', exist_ok=True)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Please enable GPU (2 T4 GPUs) in Kaggle settings'
import torch_geometric
print(f'PyG version: {torch_geometric.__version__} | device: {device}')


In [ ]:
import numpy as np
cache = torch.load(f'{RAW_CACHE_DIR}/raw_cache_25x25.pt')
patterns, s11, is_functioning = cache['patterns'], cache['s11'], cache['is_functioning']
seed_mask = np.load(f'{SEED_MASK_DIR}/seed_mask_25.npy')
print(f'Loaded {len(patterns)} samples, seed mask shape {seed_mask.shape}')


In [ ]:
import torch
from torch_geometric.data import Data
import numpy as np

def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    # Compute seed centroid
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)
    
    # Node features: (N*N + 1) nodes, 5 features each
    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])
    
    # Virtual global node (index N*N): all zeros except placeholder (-1 for is_seed)
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float)
    
    # 4-connectivity edges
    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])
    
    # Virtual node edges (connect to all metal pixels only)
    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]  # virtual edge type
    
    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    
    # Target
    y = torch.tensor(s11_db, dtype=torch.float).unsqueeze(0)  # (1, 201)
    
    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)

data = build_pyg_graph(patterns[0].numpy(), s11[0].numpy(), seed_mask, 25)
print('x shape:', data.x.shape)
print('edge_index shape:', data.edge_index.shape)
print('edge_attr shape:', data.edge_attr.shape)
print('y shape:', data.y.shape)

assert data.x.shape == (626, 5)
assert data.edge_index.shape[0] == 2
assert data.edge_attr.shape[1] == 2
assert data.edge_index.shape[1] == data.edge_attr.shape[0]
assert data.y.shape == (1, 201)


In [ ]:
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn as nn
from torch_geometric.data import Batch

class SimpleGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(5, 64)
        self.conv2 = GCNConv(64, 64)
        self.conv3 = GCNConv(64, 32)
        self.relu = nn.ReLU()
        self.out = nn.Linear(32, 201)
        
    def forward(self, x, edge_index, batch):
        x = self.relu(self.conv1(x, edge_index))
        x = self.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.out(x)

model = SimpleGCN()
batch = Batch.from_data_list([data])
out = model(batch.x, batch.edge_index, batch.batch)
print('Model output shape:', out.shape)
assert out.shape == (1, 201)


In [ ]:
from torch_geometric.data import Dataset
import os
from tqdm.auto import tqdm

class AntennaDatasetMem(Dataset):
    def __init__(self, patterns, s11, is_func, seed_mask, grid_size, proc_dir):
        self.patterns = patterns
        self.s11 = s11
        self.is_func = is_func
        self.seed_mask = seed_mask
        self.grid_size = grid_size
        self.proc_dir = proc_dir
        super().__init__(root=None, transform=None, pre_transform=None)
        
    def len(self):
        return 200  # Mini dataset
        
    def get(self, idx):
        proc_path = f'{self.proc_dir}/sample_{idx}.pt'
        if os.path.exists(proc_path):
            return torch.load(proc_path)
            
        data = build_pyg_graph(self.patterns[idx].numpy(), 
                               self.s11[idx].numpy(), 
                               self.seed_mask, self.grid_size)
        data.grid_size = self.grid_size
        data.is_functioning = int(self.is_func[idx].item())
        data.pixel_size_mm = 32.375 / self.grid_size
        torch.save(data, proc_path)
        return data

dataset = AntennaDatasetMem(patterns, s11, is_functioning, seed_mask, 25, f'{WORKING_DIR}/data/processed/25x25')

# Pre-build to ensure all 200 are saved
print("Building mini dataset...")
for idx in tqdm(range(200)):
    _ = dataset[idx]

print(f'Dataset size: {len(dataset)}')
print(f'First sample: {dataset[0]}')


In [ ]:
from torch_geometric.loader import DataLoader

loader = DataLoader(dataset, batch_size=8, shuffle=False)
batch = next(iter(loader))

print(f'Batch num_graphs: {batch.num_graphs}')
print(f'Batch x shape: {batch.x.shape}')
print(f'Batch edge_index shape: {batch.edge_index.shape}')
print(f'Batch batch shape: {batch.batch.shape}')

assert batch.batch.max().item() + 1 == 8


This notebook is verification-only — no outputs need to be transferred back. Chunk 5 rebuilds the full dataset at scale from the same raw cache.